# OpenPlaque — RCA 10–50 mm PCAT validation report

This notebook consolidates the accepted OpenPlaque RCA PCAT prototype results into one reproducible research report.

It **does not rerun centerline extraction or PCAT segmentation**. It reads the saved outputs from:
- the baseline 10–50 mm PCAT prototype,
- circular outer-wall sensitivity analysis,
- directional vessel–fat interface analysis,

and reports the primary PCAT value plus geometry sensitivity.

**Research prototype only. This is not Caristo FAI-Score and not clinical validation.**


In [ ]:
# FIRST EXECUTABLE CELL: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch pcat-validation-report-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pandas numpy matplotlib
print('Repository and packages ready.')


In [ ]:
from pathlib import Path
import shutil, zipfile, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/OpenPlaque')
BASE = ROOT / 'PCAT_RCA_10_50'
WALL = ROOT / 'PCAT_RCA_10_50_Wall_Sensitivity'
DIR = ROOT / 'PCAT_RCA_10_50_Directional_OuterWall'
OUT = ROOT / 'PCAT_RCA_10_50_Validation_Report'
OUT.mkdir(parents=True, exist_ok=True)

required = {
    'baseline_summary': BASE/'pcat_summary.csv',
    'baseline_longitudinal': BASE/'pcat_longitudinal_profile.csv',
    'baseline_radial': BASE/'pcat_radial_profile.csv',
    'wall_summary': WALL/'pcat_wall_sensitivity_summary.csv',
    'wall_stability': WALL/'pcat_wall_sensitivity_stability.csv',
    'directional_summary': DIR/'directional_pcat_summary.csv',
    'directional_longitudinal': DIR/'directional_pcat_longitudinal.csv',
    'directional_radial': DIR/'directional_pcat_radial_0p5mm.csv',
}
missing = [str(p) for p in required.values() if not p.exists()]
if missing:
    raise FileNotFoundError('Missing required saved outputs:\n' + '\n'.join(missing))

print('All required inputs found.')
print('Output:', OUT)


## 1. Load the frozen results


In [ ]:
base = pd.read_csv(required['baseline_summary'])
base_long = pd.read_csv(required['baseline_longitudinal'])
base_rad = pd.read_csv(required['baseline_radial'])
wall = pd.read_csv(required['wall_summary'])
wall_stab = pd.read_csv(required['wall_stability'])
direc = pd.read_csv(required['directional_summary'])
dir_long = pd.read_csv(required['directional_longitudinal'])
dir_rad = pd.read_csv(required['directional_radial'])

display(base.T)
display(wall)
display(direc.T)


## 2. Primary endpoint and geometry sensitivity

The baseline research endpoint is the **mean adipose attenuation in the RCA 10–50 mm segment**, using the original circular approximation with a 0.75-mm wall margin and adipose range −190 to −30 HU.

Geometry sensitivity is summarized across the five circular wall margins (0.25–1.25 mm) **plus** the directional vessel–fat interface method.


In [ ]:
b = base.iloc[0]
d = direc.iloc[0]

primary_mean = float(b['pcat_mean_hu'])
primary_median = float(b['pcat_median_hu'])
primary_sd = float(b['pcat_sd_hu'])

method_rows = []
for _, r in wall.iterrows():
    method_rows.append({
        'method': f"circular wall +{r.wall_margin_mm:.2f} mm",
        'method_class': 'circular sensitivity',
        'wall_margin_mm': float(r.wall_margin_mm),
        'pcat_mean_hu': float(r.pcat_mean_hu),
        'pcat_median_hu': float(r.pcat_median_hu),
        'fat_voxels': int(r.fat_voxels),
        'fat_volume_ml': float(r.fat_volume_ml),
        'delta_from_primary_hu': float(r.pcat_mean_hu-primary_mean),
    })
method_rows.append({
    'method': 'directional vessel-fat interface',
    'method_class': 'directional interface',
    'wall_margin_mm': np.nan,
    'pcat_mean_hu': float(d.directional_pcat_mean_hu),
    'pcat_median_hu': float(d.directional_pcat_median_hu),
    'fat_voxels': int(d.fat_voxels),
    'fat_volume_ml': float(d.fat_volume_ml),
    'delta_from_primary_hu': float(d.directional_pcat_mean_hu-primary_mean),
})
methods = pd.DataFrame(method_rows)

vals = methods.pcat_mean_hu.to_numpy(float)
geom = {
    'geometry_methods_n': len(vals),
    'geometry_mean_hu': float(np.mean(vals)),
    'geometry_median_hu': float(np.median(vals)),
    'geometry_sd_hu': float(np.std(vals, ddof=0)),
    'geometry_min_hu': float(np.min(vals)),
    'geometry_max_hu': float(np.max(vals)),
    'geometry_range_hu': float(np.ptp(vals)),
    'max_abs_delta_from_primary_hu': float(np.max(np.abs(vals-primary_mean))),
}

methods.to_csv(OUT/'openplaque_pcat_method_comparison.csv', index=False)
display(methods)
print(pd.Series(geom))


## 3. Consolidated validation summary


In [ ]:
summary = pd.DataFrame([{
    'metric_name': 'OpenPlaque PCAT Attenuation',
    'vessel': 'RCA',
    'segment_start_mm': float(b['segment_start_mm']),
    'segment_end_mm': float(b['segment_end_mm']),
    'fat_hu_low': float(b['fat_hu_low']),
    'fat_hu_high': float(b['fat_hu_high']),
    'primary_method': str(b['outer_wall_method']),
    'primary_pcat_mean_hu': primary_mean,
    'primary_pcat_median_hu': primary_median,
    'primary_pcat_sd_hu': primary_sd,
    'primary_p10_hu': float(b['pcat_p10_hu']),
    'primary_p90_hu': float(b['pcat_p90_hu']),
    'primary_fat_voxels': int(b['fat_voxels']),
    'primary_fat_volume_ml': float(b['fat_volume_ml']),
    'primary_shell_volume_ml': float(b['shell_volume_ml']),
    'primary_fat_fraction': float(b['fat_fraction_of_shell']),
    'mean_lumen_radius_mm': float(b['mean_lumen_radius_mm']),
    'mean_outer_radius_approx_mm': float(b['mean_outer_radius_approx_mm']),
    'directional_pcat_mean_hu': float(d['directional_pcat_mean_hu']),
    'directional_delta_from_primary_hu': float(d['directional_pcat_mean_hu']-primary_mean),
    'directional_median_accepted_ray_fraction': float(d['median_accepted_ray_fraction']),
    'directional_valid_centerline_fraction': float(d['valid_centerline_fraction']),
    'directional_median_wall_plus_plaque_mm': float(d['median_wall_plus_plaque_mm']),
    **geom,
    'interpretation': 'Primary mean is stable across tested geometry assumptions; radial-layer metrics remain exploratory QC.',
    'validation_scope': 'single-case research prototype; not clinical validation',
}])

summary.to_csv(OUT/'openplaque_pcat_validation_summary.csv', index=False)
display(summary.T)


## 4. Final report figures


In [ ]:
# Figure 1: geometry sensitivity
fig, ax = plt.subplots(figsize=(10,5))
x = np.arange(len(methods))
ax.plot(x, methods.pcat_mean_hu, marker='o')
ax.axhline(primary_mean, linestyle='--', label=f'primary {primary_mean:.2f} HU')
ax.set_xticks(x)
ax.set_xticklabels(methods.method, rotation=30, ha='right')
ax.set_ylabel('PCAT mean HU')
ax.set_title('OpenPlaque RCA PCAT — geometry sensitivity')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
p1 = OUT/'01_final_geometry_sensitivity.png'
fig.savefig(p1, dpi=180, bbox_inches='tight')
plt.show(); plt.close(fig)

# Figure 2: longitudinal profile comparison
fig, ax = plt.subplots(figsize=(10,5))
if {'arc_mid_mm','mean_hu'}.issubset(base_long.columns):
    ax.plot(base_long.arc_mid_mm, base_long.mean_hu, label='circular +0.75 mm')
if {'arc_start_mm','arc_end_mm','mean_hu'}.issubset(dir_long.columns):
    mid = (dir_long.arc_start_mm + dir_long.arc_end_mm)/2
    ax.plot(mid, dir_long.mean_hu, label='directional interface')
ax.axhline(primary_mean, linestyle='--', alpha=0.7)
ax.set_xlabel('RCA distance from ostium (mm)')
ax.set_ylabel('PCAT mean HU')
ax.set_title('Longitudinal RCA PCAT profile')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
p2 = OUT/'02_final_longitudinal_comparison.png'
fig.savefig(p2, dpi=180, bbox_inches='tight')
plt.show(); plt.close(fig)

# Figure 3: key radial QC comparison (exploratory only)
fig, ax = plt.subplots(figsize=(10,5))
if {'radial_mid_mm','mean_hu'}.issubset(base_rad.columns):
    ax.plot(base_rad.radial_mid_mm, base_rad.mean_hu, marker='o', label='circular +0.75 mm')
if {'radial_start_mm','radial_end_mm','mean_hu'}.issubset(dir_rad.columns):
    mid = (dir_rad.radial_start_mm + dir_rad.radial_end_mm)/2
    ax.plot(mid, dir_rad.mean_hu, marker='o', label='directional interface')
ax.set_xlabel('Distance outward from modeled boundary (mm)')
ax.set_ylabel('PCAT mean HU')
ax.set_title('Radial attenuation profile — exploratory QC only')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
p3 = OUT/'03_final_radial_qc_comparison.png'
fig.savefig(p3, dpi=180, bbox_inches='tight')
plt.show(); plt.close(fig)


## 5. Human-readable validation report


In [ ]:
s = summary.iloc[0]
report = f"""# OpenPlaque RCA PCAT validation report

## Primary research endpoint
- Metric: **OpenPlaque PCAT Attenuation**
- Vessel: **RCA**
- Longitudinal segment: **{s.segment_start_mm:.0f}–{s.segment_end_mm:.0f} mm from the ostium**
- Adipose attenuation range: **{s.fat_hu_low:.0f} to {s.fat_hu_high:.0f} HU**
- Primary PCAT mean: **{s.primary_pcat_mean_hu:.2f} HU**
- Primary median: **{s.primary_pcat_median_hu:.2f} HU**
- Primary SD: **{s.primary_pcat_sd_hu:.2f} HU**
- P10–P90: **{s.primary_p10_hu:.1f} to {s.primary_p90_hu:.1f} HU**
- Fat voxels: **{int(s.primary_fat_voxels):,}**
- Fat volume: **{s.primary_fat_volume_ml:.3f} mL**
- Shell volume: **{s.primary_shell_volume_ml:.3f} mL**
- Fat fraction of shell: **{100*s.primary_fat_fraction:.1f}%**

## Geometry sensitivity
Across five circular-wall margins (0.25–1.25 mm) plus the directional vessel–fat interface:
- Mean of method means: **{s.geometry_mean_hu:.2f} HU**
- Median of method means: **{s.geometry_median_hu:.2f} HU**
- SD across methods: **{s.geometry_sd_hu:.2f} HU**
- Full range: **{s.geometry_min_hu:.2f} to {s.geometry_max_hu:.2f} HU**
- Range width: **{s.geometry_range_hu:.2f} HU**
- Maximum absolute deviation from primary: **{s.max_abs_delta_from_primary_hu:.2f} HU**

Directional-interface result:
- PCAT mean: **{s.directional_pcat_mean_hu:.2f} HU**
- Difference from primary: **{s.directional_delta_from_primary_hu:+.2f} HU**
- Median accepted ray fraction: **{100*s.directional_median_accepted_ray_fraction:.1f}%**
- Valid centerline fraction: **{100*s.directional_valid_centerline_fraction:.1f}%**

## Interpretation
The **overall RCA 10–50 mm mean PCAT attenuation is technically stable across the tested wall/interface geometries**. A practical prototype value is therefore approximately **{s.primary_pcat_mean_hu:.1f} HU**, with geometry-method sensitivity on the order of **±{s.max_abs_delta_from_primary_hu:.1f} HU** in this case.

The radial-layer profile remains an **exploratory QC output**, not a validated biological endpoint, because the inferred vessel–fat boundary is model-dependent.

## Scope
This is a **single-case OpenPlaque research prototype validation**, not clinical validation and not Caristo FAI-Score.
"""

report_path = OUT/'OPENPLAQUE_PCAT_VALIDATION_REPORT.md'
report_path.write_text(report)
print(report)


## 6. Package everything to report back


In [ ]:
# Copy source QC figures when present.
source_figs = [
    BASE/'01_pcat_axial_qc.png',
    BASE/'02_pcat_profiles.png',
    WALL/'01_wall_sensitivity_summary.png',
    WALL/'02_radial_profiles_by_margin.png',
    DIR/'01_directional_interface_qc.png',
    DIR/'02_directional_summary.png',
    DIR/'03_directional_composition.png',
]
for p in source_figs:
    if p.exists():
        shutil.copy2(p, OUT/p.name)

zip_path = OUT/'OPENPLAQUE_PCAT_VALIDATION_REPORT_BACK.zip'
members = [
    OUT/'openplaque_pcat_validation_summary.csv',
    OUT/'openplaque_pcat_method_comparison.csv',
    OUT/'OPENPLAQUE_PCAT_VALIDATION_REPORT.md',
    OUT/'01_final_geometry_sensitivity.png',
    OUT/'02_final_longitudinal_comparison.png',
    OUT/'03_final_radial_qc_comparison.png',
]
members += [OUT/p.name for p in source_figs if (OUT/p.name).exists()]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in members:
        z.write(p, arcname=p.name)

print('Created:', zip_path)
print()
print('Direct Drive search URLs:')
for name in [
    'OPENPLAQUE_PCAT_VALIDATION_REPORT_BACK.zip',
    'OPENPLAQUE_PCAT_VALIDATION_REPORT.md',
    'openplaque_pcat_validation_summary.csv',
    'openplaque_pcat_method_comparison.csv',
    '01_final_geometry_sensitivity.png',
    '02_final_longitudinal_comparison.png',
    '03_final_radial_qc_comparison.png',
]:
    print(f'https://drive.google.com/drive/u/0/search?q={name}')
